# GeoModel-LLM: training and 3D inference

This notebook uses the supplied training, validation, test, and engineering-case JSONL files. Visualization and metric computation have been removed. Sections are disabled by default; enable only the intended operation on a suitable CUDA machine. No training or inference was run while preparing this release.


## 1. Configuration

Launch Jupyter in this folder, or set GEOMODEL_ROOT. The three supplied synthetic splits are used directly.


In [ ]:
import os
from pathlib import Path

REPO_ROOT = Path(os.environ.get("GEOMODEL_ROOT", str(Path.cwd()))).resolve()
DATA_DIR = Path(os.environ.get("GEOMODEL_DATA_DIR", str(REPO_ROOT)))
OUTPUT_DIR = REPO_ROOT / "outputs"
ADAPTER_DIR = OUTPUT_DIR / "adapter"
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
INFERENCE_DIR = OUTPUT_DIR / "inference"

BASE_MODEL = os.environ.get("GEOMODEL_BASE_MODEL", "Qwen/Qwen3-14B")
ADAPTER_SOURCE = os.environ.get("GEOMODEL_ADAPTER", "js123js/Qwen3_14_finetune")
TRAIN_FILE = DATA_DIR / "finetune_data_3d_train.jsonl"
VALIDATION_FILE = DATA_DIR / "finetune_data_3d_validation.jsonl"
TEST_FILE = DATA_DIR / "finetune_data_3d_test.jsonl"
ENGINEERING_FILE = DATA_DIR / "instruction_input_largerscale.jsonl"

# Enable only the sections you intend to execute on a CUDA-capable machine.
RUN_TRAINING = False
RUN_SYNTHETIC_INFERENCE = False
RUN_ENGINEERING_INFERENCE = False


In [ ]:
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""


## 2. Training

Set RUN_TRAINING=True. This section trains with the supplied train split, monitors validation loss on the supplied validation split, and saves the adapter. The test file is not used for fitting.


In [ ]:
if RUN_TRAINING:
    from unsloth import FastLanguageModel
    from datasets import load_dataset
    from trl import SFTTrainer, SFTConfig
    from unsloth.chat_templates import train_on_responses_only

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=BASE_MODEL,
        max_seq_length=4000,
        load_in_4bit=True,
        load_in_8bit=False,
        full_finetuning=False,
        disable_log_stats=True,
    )
    model = FastLanguageModel.get_peft_model(
        model,
        r=64,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"],
        lora_alpha=128,
        lora_dropout=0,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=3407,
        use_rslora=False,
        loftq_config=None,
    )


In [ ]:
if RUN_TRAINING:
    # Use the author's supplied split directly; do not randomly split again.
    train_dataset = load_dataset("json", data_files=str(TRAIN_FILE), split="train")
    validation_dataset = load_dataset("json", data_files=str(VALIDATION_FILE), split="train")

    def format_training_rows(examples):
        texts = [
            alpaca_prompt.format(instruction, input_text, output) + tokenizer.eos_token
            for instruction, input_text, output in zip(
                examples["instruction"], examples["input"], examples["output"]
            )
        ]
        return {"text": texts}

    train_dataset = train_dataset.map(format_training_rows, batched=True)
    validation_dataset = validation_dataset.map(format_training_rows, batched=True)


In [ ]:
if RUN_TRAINING:
    CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_dataset,
        eval_dataset=validation_dataset,
        args=SFTConfig(
            dataset_text_field="text",
            per_device_train_batch_size=8,
            gradient_accumulation_steps=8,
            warmup_steps=50,
            num_train_epochs=2,
            learning_rate=2e-4,
            logging_steps=1,
            optim="adamw_8bit",
            weight_decay=0.001,
            lr_scheduler_type="cosine",
            seed=3407,
            report_to="none",
            run_name="geo_3d_run_v1",
            eval_strategy="steps",
            eval_steps=40,
            save_strategy="steps",
            save_steps=40,
            save_total_limit=2,
            output_dir=str(CHECKPOINT_DIR),
        ),
    )
    trainer = train_on_responses_only(
        trainer,
        instruction_part="### Instruction:\n",
        response_part="### Response:\n",
    )
    trainer_stats = trainer.train()


In [ ]:
if RUN_TRAINING:
    ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(str(ADAPTER_DIR))
    tokenizer.save_pretrained(str(ADAPTER_DIR))
    # Subsequent inference in this session uses the newly trained local adapter.
    ADAPTER_SOURCE = str(ADAPTER_DIR)


## 3. Load the inference model

For inference only, leave RUN_TRAINING=False and enable the relevant inference flag. The published adapter is the default; GEOMODEL_ADAPTER may specify another adapter or a local directory. The base model is loaded explicitly rather than following a private path in an older adapter configuration.


In [ ]:
if RUN_SYNTHETIC_INFERENCE or RUN_ENGINEERING_INFERENCE:
    from unsloth import FastLanguageModel
    from peft import PeftModel
    from transformers import AutoTokenizer
    import torch

    # Load the base explicitly: a legacy adapter config may contain a private
    # server path that should not be used to locate the base model.
    base_inference_model, _ = FastLanguageModel.from_pretrained(
        model_name=BASE_MODEL,
        max_seq_length=4000,
        load_in_4bit=True,
    )
    inference_model = PeftModel.from_pretrained(base_inference_model, ADAPTER_SOURCE)
    inference_tokenizer = AutoTokenizer.from_pretrained(ADAPTER_SOURCE)
    FastLanguageModel.for_inference(inference_model)
    inference_model.eval()
    inference_tokenizer.padding_side = "left"
    if inference_tokenizer.pad_token is None:
        inference_tokenizer.pad_token = inference_tokenizer.eos_token
    INFERENCE_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
import json
import re


def load_jsonl(path):
    with path.open("r", encoding="utf-8-sig") as handle:
        return [json.loads(line) for line in handle if line.strip()]


def target_x(row):
    match = re.search(r"\bX\s*=\s*(\d+)", row["instruction"], re.I)
    if match is None:
        raise ValueError("Missing target X index in the instruction.")
    return int(match.group(1))


def borehole_context(row):
    return row["input"].split("Target:", 1)[0].strip()


def split_complete_models(rows, nx):
    # Reconstruct complete cubes from the new test file, using its own row
    # order rather than indices into the old 800-model dataset.
    if len(rows) % nx:
        raise ValueError("The JSONL does not contain complete groups of x slices.")
    groups = []
    for start in range(0, len(rows), nx):
        group = rows[start:start + nx]
        if [target_x(row) for row in group] != list(range(nx)):
            raise ValueError("Expected ordered X=0,...,nx-1 within each model.")
        if len({borehole_context(row) for row in group}) != 1:
            raise ValueError("Borehole inputs differ within a proposed model group.")
        groups.append(group)
    return groups


def decode_column(rle_text, depth):
    # Retain the original parser's padding/truncation behavior.
    import numpy as np
    values = []
    for part in rle_text.split(","):
        part = part.strip()
        try:
            if "x" in part:
                count, label = part.split("x")
                values.extend([float(label)] * int(count))
            else:
                values.append(float(part))
        except (ValueError, TypeError):
            continue
    if not values:
        return np.zeros(depth)
    return np.asarray((values + [values[-1]] * max(0, depth - len(values)))[:depth])


def infer_cube(rows, shape, batch_size, max_new_tokens):
    import numpy as np
    import torch
    # Original behavior: unmatched/missing columns remain initialized to zero.
    cube = np.zeros(shape)
    expected_x = [target_x(row) for row in rows]
    if expected_x != list(range(shape[0])):
        raise ValueError("A complete ordered set of x slices is required.")
    pattern = re.compile(r"(?:Col)?\[?(\d+),\s*(\d+)\]?:\s*\[(.*?)\]")
    for start in range(0, len(rows), batch_size):
        batch = rows[start:start + batch_size]
        prompts = [alpaca_prompt.format(row["instruction"], row["input"], "") for row in batch]
        inputs = inference_tokenizer(
            prompts, return_tensors="pt", padding=True,
            truncation=True, max_length=2048,
        ).to("cuda")
        prompt_length = inputs["input_ids"].shape[1]
        with torch.no_grad():
            outputs = inference_model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                use_cache=True,
                do_sample=False,
                pad_token_id=inference_tokenizer.pad_token_id,
            )
        texts = inference_tokenizer.batch_decode(
            outputs[:, prompt_length:], skip_special_tokens=True,
        )
        for text in texts:
            text = text.split("<|", 1)[0].split("###", 1)[0].strip()
            for x_text, y_text, rle_text in pattern.findall(text):
                x, y = int(x_text), int(y_text)
                if x >= shape[0] or y >= shape[1]:
                    continue
                cube[x, y, :] = decode_column(rle_text, shape[2])
        print(f"Generated {min(start + batch_size, len(rows))}/{len(rows)} x slices")
    return cube


## 4. Synthetic test-set 3D inference

Set RUN_SYNTHETIC_INFERENCE=True. Each complete group of 40 ordered x slices is reconstructed into a 40x50x30 prediction. Sequential output IDs refer to this supplied test file, not the old 800-model dataset. No accuracy or other evaluation metrics are computed.


In [ ]:
if RUN_SYNTHETIC_INFERENCE:
    import numpy as np
    test_rows = load_jsonl(TEST_FILE)
    test_models = split_complete_models(test_rows, nx=40)
    for model_number, model_rows in enumerate(test_models, start=1):
        cube = infer_cube(model_rows, shape=(40, 50, 30), batch_size=40, max_new_tokens=4000)
        output_file = INFERENCE_DIR / f"cube_pred_test_{model_number:03d}.npy"
        np.save(output_file, cube)
        print(f"Saved {output_file.name}")


## 5. Engineering-case 3D inference

Set RUN_ENGINEERING_INFERENCE=True. The supplied case contains 60 x slices and produces one 60x100x40 prediction. Only instruction and input fields are required.


In [ ]:
if RUN_ENGINEERING_INFERENCE:
    import numpy as np
    engineering_rows = load_jsonl(ENGINEERING_FILE)
    engineering_models = split_complete_models(engineering_rows, nx=60)
    if len(engineering_models) != 1:
        raise ValueError("Expected one engineering case containing 60 ordered x slices.")
    cube = infer_cube(
        engineering_models[0], shape=(60, 100, 40), batch_size=20, max_new_tokens=8000,
    )
    output_file = INFERENCE_DIR / "cube_pred_largerscale.npy"
    np.save(output_file, cube)
    print(f"Saved {output_file.name}")
